# Notebook 01 — GEE Data Preprocessing
**Project:** WASHLAB Climate-Smart WASH Pilot — Kitui County  
**Analyst:** Davis Mironga  
**Purpose:** Pull and export all satellite datasets from Google Earth Engine for use in Colab notebooks.  
**Output:** Preprocessed rasters exported to Google Drive.

---
## Datasets pulled
| Dataset | Source | Resolution | Use |
|---------|--------|------------|-----|
| NDVI | MODIS MOD13A3 | 1km monthly | Vegetation condition |
| Rainfall | CHIRPS v2.0 | 5km monthly | Seasonal water availability |
| Land Surface Temp | MODIS MOD11A2 | 1km 8-day | Heat and aridity |
| Evapotranspiration | MODIS MOD16A2 | 500m 8-day | Water consumption |
| Terrain / DEM | SRTM 30m | 30m | Slope and accessibility |
| Population density | WorldPop 2020 | 100m | Exposure weighting |
| Soil moisture | ERA5-Land | ~9km monthly | Groundwater recharge proxy |
| Surface water | JRC Global Surface Water | 30m | Permanent vs seasonal |

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
!pip install earthengine-api geemap geopandas -q

import ee
import geemap
import geopandas as gpd
from google.colab import drive

drive.mount('/content/drive')
ee.Authenticate()
ee.Initialize(project='your-gee-project-id')  # TODO: replace with your GEE project ID

DRIVE_OUT = '/content/drive/MyDrive/Kitui_WASHLAB/satellite/'
print('Setup complete')

In [ ]:
# ── 1. Define Kitui County study area ─────────────────────────────────────────
# Load Kitui boundary from GADM via GEE
kenya = ee.FeatureCollection('FAO/GAUL/2015/level1')
kitui = kenya.filter(ee.Filter.eq('ADM1_NAME', 'Kitui'))
kitui_geom = kitui.geometry()

# Verify boundary
Map = geemap.Map()
Map.centerObject(kitui_geom, 8)
Map.addLayer(kitui_geom, {'color': '0B5394'}, 'Kitui County')
Map

In [ ]:
# ── 2. NDVI — MODIS MOD13A3 (2000–2025) ──────────────────────────────────────
NDVI_START = '2000-01-01'
NDVI_END   = '2025-12-31'

ndvi = (ee.ImageCollection('MODIS/061/MOD13A3')
          .filterDate(NDVI_START, NDVI_END)
          .filterBounds(kitui_geom)
          .select('NDVI')
          .map(lambda img: img.multiply(0.0001)  # scale factor
                              .copyProperties(img, ['system:time_start'])))

# Annual mean NDVI
years = ee.List.sequence(2000, 2025)
def annual_ndvi(year):
    y = ee.Number(year).int()
    return (ndvi.filter(ee.Filter.calendarRange(y, y, 'year'))
                .mean()
                .set('year', y)
                .set('system:time_start', ee.Date.fromYMD(y, 1, 1).millis()))

ndvi_annual = ee.ImageCollection(years.map(annual_ndvi))
print('NDVI collection ready:', ndvi_annual.size().getInfo(), 'annual images')

In [ ]:
# ── 3. Rainfall — CHIRPS v2.0 (1981–2025) ────────────────────────────────────
chirps = (ee.ImageCollection('UCSB-CHG/CHIRPS/PENTAD')
            .filterDate('1981-01-01', '2025-12-31')
            .filterBounds(kitui_geom)
            .select('precipitation'))

# Annual total rainfall
def annual_rainfall(year):
    y = ee.Number(year).int()
    return (chirps.filter(ee.Filter.calendarRange(y, y, 'year'))
                  .sum()
                  .set('year', y)
                  .set('system:time_start', ee.Date.fromYMD(y, 1, 1).millis()))

years_full = ee.List.sequence(1981, 2025)
rainfall_annual = ee.ImageCollection(years_full.map(annual_rainfall))

# Long-term mean (1981–2010 baseline)
rainfall_baseline = (chirps.filterDate('1981-01-01', '2010-12-31')
                           .sum()
                           .divide(30)  # annual mean
                           .rename('rainfall_baseline_mm_yr'))

# Seasonal: short rains (Oct–Dec) vs long rains (Mar–May)
short_rains = (chirps.filter(ee.Filter.calendarRange(10, 12, 'month'))
                     .mean().rename('short_rains_mean'))
long_rains  = (chirps.filter(ee.Filter.calendarRange(3, 5, 'month'))
                     .mean().rename('long_rains_mean'))

print('Rainfall layers ready')

In [ ]:
# ── 4. Terrain — SRTM 30m ─────────────────────────────────────────────────────
srtm  = ee.Image('USGS/SRTMGL1_003').clip(kitui_geom)
slope = ee.Terrain.slope(srtm).rename('slope_deg')
elev  = srtm.rename('elevation_m')

# Flow accumulation proxy for watershed delineation (optional)
print('Terrain ready')

In [ ]:
# ── 5. Population — WorldPop 2020 ─────────────────────────────────────────────
pop = (ee.ImageCollection('WorldPop/GP/100m/pop')
         .filter(ee.Filter.eq('country', 'KEN'))
         .filter(ee.Filter.eq('year', 2020))
         .first()
         .clip(kitui_geom)
         .rename('population_100m'))
print('Population ready')

In [ ]:
# ── 6. JRC Surface Water ──────────────────────────────────────────────────────
jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').clip(kitui_geom)

# Seasonality band: 0=no water, 1=seasonal, 12=permanent
water_seasonality = jrc.select('seasonality').rename('water_seasonality')
# Occurrence: % of time water present
water_occurrence  = jrc.select('occurrence').rename('water_occurrence_pct')
print('JRC Surface Water ready')

In [ ]:
# ── 7. Land Surface Temperature — MODIS MOD11A2 ───────────────────────────────
lst = (ee.ImageCollection('MODIS/061/MOD11A2')
         .filterDate('2000-01-01', '2025-12-31')
         .filterBounds(kitui_geom)
         .select('LST_Day_1km')
         .map(lambda img: img.multiply(0.02).subtract(273.15)  # Kelvin to Celsius
                             .copyProperties(img, ['system:time_start'])))

lst_mean = lst.mean().clip(kitui_geom).rename('lst_mean_celsius')
print('LST ready')

In [ ]:
# ── 8. Export all layers to Google Drive ──────────────────────────────────────
export_params = {
    'region': kitui_geom,
    'scale': 500,
    'crs': 'EPSG:4326',
    'folder': 'Kitui_WASHLAB/satellite',
    'maxPixels': 1e13,
    'fileFormat': 'GeoTIFF',
}

exports = [
    (ndvi_annual.mean().clip(kitui_geom).rename('ndvi_mean'),     'kitui_ndvi_mean_2000_2025'),
    (rainfall_baseline,                                            'kitui_rainfall_baseline_1981_2010'),
    (short_rains,                                                  'kitui_short_rains_mean'),
    (long_rains,                                                   'kitui_long_rains_mean'),
    (elev,                                                         'kitui_elevation_srtm30'),
    (slope,                                                        'kitui_slope_deg'),
    (pop,                                                          'kitui_worldpop_2020'),
    (water_seasonality,                                            'kitui_jrc_water_seasonality'),
    (water_occurrence,                                             'kitui_jrc_water_occurrence'),
    (lst_mean,                                                     'kitui_lst_mean_celsius'),
]

tasks = []
for image, filename in exports:
    task = ee.batch.Export.image.toDrive(
        image=image,
        description=filename,
        fileNamePrefix=filename,
        **export_params
    )
    task.start()
    tasks.append(task)
    print(f'Export started: {filename}')

print(f'\n{len(tasks)} export tasks submitted. Check GEE Task Manager.')
print('Exports will take 5–30 minutes depending on size.')